In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (confusion_matrix,accuracy_score,
recall_score,f1_score,roc_auc_score,precision_score)

: 

In [ ]:
df=pd.read_csv(r'C:\Users\USER\Desktop\5th sem\project1_daibatise\data\processed\mohammed_ajlan_p1w1_features.csv')

In [ ]:
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]
x = df[feature_cols].copy()
y = df["outcome"]
x

In [ ]:
x=pd.get_dummies(x,drop_first=True)
x.shape

In [ ]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)



In [ ]:
def score_row(name, pred,scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    r = {
        "model":name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp)}
    if scores is  not None :
     r["auc"] = round(roc_auc_score(y_test,scores),4)
     return r
    print("Train:", X_train.shape, " Test:", X_test.shape)
print("Diabetic patients in the test set:", int(np.sum(y_test)), "of", len(y_test))

for nm, m, Xt in [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]:
    print(f"{nm:22s} accuracy {accuracy_score(y_test, m.predict(Xt)):.4f}")

restate the target you are aiming at


In [ ]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182

print("PROJECT METRIC : recall  (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR, "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

SVM (Support Vector Machine) is a machine learning algorithm used mainly for classification.
It finds the best boundary (hyperplane) that separates different classes of data with the maximum possible margin between them.

train an svm with default settings

In [ ]:
from sklearn.svm import SVC

svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled, y_train)

pred_rbf = svm_rbf.predict(x_test_scaled)

print("Fitted:", svm_rbf)
print("Patients flagged:", int(pred_rbf.sum()), "of", len(pred_rbf))

In [ ]:
row_rbf= score_row("SVC rbf (default)",pred_rbf,
                   svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print("target recall:", TARGET_RECALL,"| best so far:",BEST_SO_FAR)
print("this model :", row_rbf["recall"])

try a straight boundary instead

In [ ]:
svm_liner = SVC(kernel='linear',random_state=42)
svm_liner.fit(x_train_scaled,y_train)                       
pred_liner = svm_liner.predict(x_test_scaled)
print("fitted:", svm_liner)
print("patients flagged:",int(pred_liner.sum()), "of", len(pred_liner))


In [ ]:
row_liner = score_row('SVC linear', pred_liner,svm_liner.decision_function(x_test_scaled))
print(pd.DataFrame([row_liner,row_rbf]).to_string(index=False))

look closely at what the linear svm predicted

In [ ]:
print("Patients the linear SVM flagged as diabetic:", int(pred_liner.sum()), "of", len(pred_liner))
print()
print("Linear SVM confusion matrix:")
print(confusion_matrix(y_test, pred_liner))
print()
print("Week 5 dummy model confusion matrix:*")
print(confusion_matrix(y_test, np.zeros_like(y_test)))
print()
print("Linear SVM accuracy :", round(accuracy_score(y_test, pred_liner), 4))
print("Dummy model accuracy:", round(accuracy_score(y_test, np.zeros_like(y_test)), 4))


learn what class_weight does?

Class weight tells a machine learning model to give more importance to certain classes, usually the minority class. It helps the model handle imbalanced datasets by penalizing mistakes on important or underrepresented classes more heavily.

refit both svms with balanced class weights

In [ ]:
svm_rbf_bal = SVC(random_state=42, class_weight="balanced").fit(
    x_train_scaled, y_train
)

svm_lin_bal = SVC(kernel="linear", random_state=42,
                  class_weight="balanced").fit(
    x_train_scaled, y_train
)

rows = [
    row_rbf,
    score_row(
        "SVC rbf (balanced)",
        svm_rbf_bal.predict(x_test_scaled),
        svm_rbf_bal.decision_function(x_test_scaled)
    ),
    row_liner,
    score_row(
        "SVC linear (balanced)",
        svm_lin_bal.predict(x_test_scaled),
        svm_lin_bal.decision_function(x_test_scaled)
    )
]

svm_table = pd.DataFrame(rows)

print(svm_table.to_string(index=False))

get a risk score out of the svm

In [ ]:
scores_bal= svm_rbf_bal.decision_function(x_test_scaled)
print("decision_function output for the first 8 test patients:")
print(np.round(scores_bal[:8], 3))

print("negative = healthy side of the boundary, positive = diabetic side")

print()

print("AUC from decision_function:", 
      round(roc_auc_score(y_test, scores_bal), 4))

print("AUC of the linear SVM that found nobody:",
      round(roc_auc_score(y_test, svm_liner.decision_function(x_test_scaled)), 4))

# RANDOM FOREST

EXPLAIN WHY MANY TREES BEAT ONE

Random Forest is generally better than a Decision Tree because it combines multiple decision trees to make predictions. It reduces overfitting, improves stability, and often provides better accuracy on unseen data. However, it requires more computational resources and is harder to interpret

Explain what makes the trees different from each other

Random Forest trees differ from each other because each tree is trained using different random samples of the data and different subsets of features. This diversity helps the trees make different predictions, which are combined to produce the final result.

main 2 sources 

Random Sampling of Data: Each decision tree is trained using a randomly selected sample of the training dataset.

Random Selection of Features: At each split, the tree considers a random subset of features to decide the best split.

Train a random forest with default settings

In [ ]:
from sklearn.ensemble import RandomForestClassifier